In [1]:
# 1. Cài đặt các thư viện Python cần thiết
!pip install -q selenium webdriver-manager bs4 requests pandas urllib3

# 2. Thêm key và repository chính thức của Google Chrome
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'

# 3. Cập nhật apt và cài đặt Google Chrome Stable
!apt-get update -y -q
!apt-get install -y google-chrome-stable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,213 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 http://se

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime
import pandas as pd
import os

# --- Cài đặt Selenium ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- CẤU HÌNH CƠ BẢN ---
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

START_YEAR = 2023  # Bot sẽ cào dữ liệu từ năm này trở đi
END_YEAR = 2025

# Danh sách chuyên mục (Bạn có thể thêm bớt tùy ý dựa trên file JSON của bạn)
CATEGORIES = {
    "thoi su": {
        "phap luat": "https://thanhnien.vn/thoi-su/phap-luat.htm",
        "dan sinh": "https://thanhnien.vn/thoi-su/dan-sinh.htm",
        "lao đong - viec lam": "https://thanhnien.vn/thoi-su/lao-dong-viec-lam.htm",
        "quyen đuoc biet": "https://thanhnien.vn/thoi-su/quyen-duoc-biet.htm",
        "phong su / đieu tra": "https://thanhnien.vn/thoi-su/phong-su--dieu-tra.htm",
        "quoc phong": "https://thanhnien.vn/thoi-su/quoc-phong.htm",
        "chong tin gia": "https://thanhnien.vn/thoi-su/chong-tin-gia.htm",
        "thanh tuu y khoa": "https://thanhnien.vn/thoi-su/thanh-tuu-y-khoa.htm"
    },
    "the gioi": {
        "kinh te the gioi": "https://thanhnien.vn/the-gioi/kinh-te-the-gioi.htm",
        "quan su": "https://thanhnien.vn/the-gioi/quan-su.htm",
        "goc nhin": "https://thanhnien.vn/the-gioi/goc-nhin.htm",
        "ho so": "https://thanhnien.vn/the-gioi/ho-so.htm",
        "nguoi viet nam chau": "https://thanhnien.vn/the-gioi/nguoi-viet-nam-chau.htm",
        "chuyen la": "https://thanhnien.vn/the-gioi/chuyen-la.htm"
    },
    "kinh te": {
        "kinh te xanh": "https://thanhnien.vn/kinh-te/kinh-te-xanh.htm",
        "chinh sach - phat trien": "https://thanhnien.vn/kinh-te/chinh-sach-phat-trien.htm",
        "ngan hang": "https://thanhnien.vn/kinh-te/ngan-hang.htm",
        "chung khoan": "https://thanhnien.vn/kinh-te/chung-khoan.htm",
        "doanh nghiep": "https://thanhnien.vn/kinh-te/doanh-nghiep.htm",
        "khat vong viet nam": "https://thanhnien.vn/kinh-te/doanh-nhan.htm",
        "lam giau": "https://thanhnien.vn/kinh-te/lam-giau.htm",
        "đia oc": "https://thanhnien.vn/kinh-te/dia-oc.htm"
    },
    "đoi song": {
        "thanh nien va toi": "https://thanhnien.vn/doi-song/thanh-nien-va-toi.htm",
        "tet yeu thuong": "https://thanhnien.vn/doi-song/tet-yeu-thuong.htm",
        "nguoi song quanh ta": "https://thanhnien.vn/doi-song/nguoi-song-quanh-ta.htm",
        "gia đinh": "https://thanhnien.vn/doi-song/gia-dinh.htm",
        "am thuc": "https://thanhnien.vn/doi-song/am-thuc.htm",
        "cong đong": "https://thanhnien.vn/doi-song/cong-dong.htm",
        "mot nua the gioi": "https://thanhnien.vn/doi-song/mot-nua-the-gioi.htm",
        "khat vong nam rong": "https://thanhnien.vn/doi-song/khat-vong-nam-rong.htm"
    },
    "suc khoe": {
        "khoe đep moi ngay": "https://thanhnien.vn/suc-khoe/khoe-dep-moi-ngay.htm",
        "lam đep": "https://thanhnien.vn/suc-khoe/lam-dep.htm",
        "gioi tinh": "https://thanhnien.vn/suc-khoe/gioi-tinh.htm",
        "y te thong minh": "https://thanhnien.vn/suc-khoe/y-te-thong-minh.htm",
        "tham my an toan": "https://thanhnien.vn/suc-khoe/tham-my-an-toan.htm",
        "tin hay y te": "https://thanhnien.vn/suc-khoe/tin-hay-y-te.htm"
    },
    "giao duc": {
        "tuyen sinh": "https://thanhnien.vn/giao-duc/tuyen-sinh.htm",
        "chon nghe - chon truong": "https://thanhnien.vn/giao-duc/chon-nghe-chon-truong.htm",
        "du hoc": "https://thanhnien.vn/giao-duc/du-hoc.htm",
        "nha truong": "https://thanhnien.vn/giao-duc/nha-truong.htm",
        "phu huynh": "https://thanhnien.vn/giao-duc/phu-huynh.htm",
        "tra cuu điem thi": "https://thanhnien.vn/giao-duc/tra-cuu-diem-thi.htm",
        "cam nang tuyen sinh 2026": "https://camnangtuyensinh.thanhnien.vn",
        "on thi tot nghiep": "https://thanhnien.vn/giao-duc/on-thi-tot-nghiep.htm"
    },
    "du lich": {
        "tin tuc - su kien": "https://thanhnien.vn/du-lich/tin-tuc-su-kien.htm",
        "choi gi, an đau, đi the nao?": "https://thanhnien.vn/du-lich/choi-gi-an-dau-di-the-nao.htm",
        "bat đong san du lich": "https://thanhnien.vn/du-lich/bat-dong-san-du-lich.htm",
        "cau chuyen du lich": "https://thanhnien.vn/du-lich/cau-chuyen-du-lich.htm",
        "kham pha": "https://thanhnien.vn/du-lich/kham-pha.htm"
    },
    "giai tri": {
        "ket noi": "https://ketnoi.thanhnien.vn/",
        "phim": "https://thanhnien.vn/giai-tri/phim.htm",
        "truyen hinh": "https://thanhnien.vn/giai-tri/truyen-hinh.htm",
        "đoi nghe si": "https://thanhnien.vn/giai-tri/doi-nghe-si.htm"
    },
    "cong nghe": {
        "tin tuc cong nghe": "https://thanhnien.vn/cong-nghe/tin-tuc-cong-nghe.htm",
        "blockchain": "https://thanhnien.vn/cong-nghe/blockchain.htm",
        "san pham": "https://thanhnien.vn/cong-nghe/san-pham.htm",
        "xu huong - chuyen đoi so": "https://thanhnien.vn/cong-nghe/xu-huong-chuyen-doi-so.htm",
        "thu thuat": "https://thanhnien.vn/cong-nghe/thu-thuat.htm",
        "game": "https://thanhnien.vn/cong-nghe/game.htm"
    },
    "xe": {
        "thi truong": "https://thanhnien.vn/xe/thi-truong.htm",
        "xe xanh": "https://thanhnien.vn/xe/xe-xanh.htm",
        "đanh gia xe": "https://thanhnien.vn/xe/danh-gia-xe.htm",
        "tu van": "https://thanhnien.vn/xe/tu-van.htm",
        "video": "https://thanhnien.vn/xe/video.htm",
        "dien đan": "https://thanhnien.vn/xe/dien-dan.htm",
        "xe - giao thong": "https://thanhnien.vn/xe/xe-giao-thong.htm"
    }
}

def get_article_year(url):
    """Hàm phụ: Đọc nhanh ngày xuất bản của 1 bài viết để check mốc thời gian"""
    try:
        res = requests.get(url, headers=HEADERS, timeout=5)
        soup = BeautifulSoup(res.text, 'html.parser')
        date_tag = soup.find('meta', property='article:published_time')
        if date_tag:
            # Dữ liệu mẫu: 2026-04-24T16:10:00+07:00
            pub_date_str = date_tag.get('content')
            return datetime.strptime(pub_date_str[:10], "%Y-%m-%d").year
    except:
        pass
    return None

def setup_driver():
    """Khởi tạo trình duyệt ẩn danh được tối ưu cực nhẹ, chống treo"""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage") # Bắt buộc trên Kaggle/Colab để không tràn RAM
    options.add_argument("--disable-gpu") # Tắt GPU acceleration
    options.add_argument("--blink-settings=imagesEnabled=false") # Tắt tải hình ảnh để siêu tốc
    
    # CHIẾN THUẬT QUAN TRỌNG: Chỉ đợi HTML tải xong, không đợi Ads/Ảnh
    options.page_load_strategy = 'eager' 
    
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # Ép thời gian tải trang tối đa là 45 giây (Nếu quá sẽ quăng lỗi để code xử lý chứ không treo)
    driver.set_page_load_timeout(45) 
    return driver


def get_all_links(category_url):    
    """Thuật toán Auto-Scroll dùng Javascript siêu tốc"""
    driver = setup_driver()
    print(f"  [*] Đang quét link tại: {category_url}")

    try:
        driver.get(category_url)
    except Exception:
        print(f"    [!] Trang tải hơi chậm, ép buộc bỏ qua khâu chờ...")
        try:
            driver.execute_script("window.stop();")
        except:
            pass

    all_links = set()
    prev_article_count = 0
    stuck_attempts = 0

    while True:
        # 1. DÙNG JAVASCRIPT GOM LINK (Nhanh và chính xác tuyệt đối)
        js_get_links = """
            let results = [];
            // Quét mọi khối bài báo trên trang
            let articles = document.querySelectorAll('.box-category-item, .item-first, .box-news, article');
            
            articles.forEach(art => {
                // closest() giúp check chính xác khối này có nằm trong sidebar không
                if(art.closest('.section__stream-sub-cate, .list__stream-sub, .box-right, .list-box-right, .box-category-sub')) {
                    return; // Bỏ qua sidebar
                }
                
                // Tìm thẻ a chứa link
                let aTag = art.querySelector('h3 a, h2 a, a.box-category-link-title, .box-title a, .title-news a');
                // Link hợp lệ phải kết thúc bằng .htm (né link tag, link category)
                if(aTag && aTag.href && aTag.href.includes('.htm')) {
                    results.push(aTag.href);
                }
            });
            return results;
        """
        
        # Lấy danh sách link từ trình duyệt
        extracted_urls = driver.execute_script(js_get_links)
        
        # Chỉ check những link thực sự MỚI xuất hiện trong lần cuộn này
        new_links_this_scroll = []
        for url in extracted_urls:
            if url not in all_links:
                all_links.add(url)
                new_links_this_scroll.append(url)
                
        current_count = len(all_links)

        # --- BỘ LỌC THỜI GIAN ---
        if new_links_this_scroll:
            # Lấy 3 link mới nhất ở đáy trang để check năm
            recent_urls = new_links_this_scroll[-3:] 
            oldest_year_in_view = 9999
            
            for url in recent_urls:
                year = get_article_year(url)
                if year:
                    if year < oldest_year_in_view:
                        oldest_year_in_view = year
                else:
                    print(f"    [?] Không đọc được năm của link: {url[-40:]}")
            
            print(f"    [Check] Năm cũ nhất đáy trang: {oldest_year_in_view} | Tổng link đã gom: {current_count}")
            
            if oldest_year_in_view != 9999 and oldest_year_in_view < START_YEAR:
                print(f"    [Stop] Đã chạm mốc năm {oldest_year_in_view}. Ngừng cuộn.")
                break

        # --- BỘ CHỐNG KẸT ---
        if current_count == prev_article_count:
            stuck_attempts += 1
            if stuck_attempts >= 4:
                print("    [Warning] Không có thêm link mới sau 4 lần lướt (Chạm đáy). Buộc ngắt!")
                break
        else:
            stuck_attempts = 0 
            prev_article_count = current_count

        # 3. KÉO XUỐNG & CLICK "XEM THÊM" BẰNG JAVASCRIPT
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight - 900);")
            time.sleep(1.5) # Rút ngắn thời gian chờ cuộn

            js_click = """
                let btns = document.querySelectorAll("a.list__viewmore, a.view-more, a[class*='view-more']");
                if(btns.length > 0) {
                    btns[btns.length - 1].click();
                    return true;
                }
                return false;
            """
            clicked = driver.execute_script(js_click)
            
            if not clicked:
                print("    [Info] Không tìm thấy nút Xem thêm (Đã hết trang).")
                break
                
            time.sleep(2.5) # Chờ bài mới load về
        except Exception:
            break

    driver.quit()
    return list(all_links)
     

def fetch_detail(url, category, sub_topic):
    """Bóc tách dữ liệu theo đúng cấu trúc Meta Tags của Báo Thanh Niên"""
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')

        # Hàm lambda để lấy nội dung thẻ Meta cho gọn
        get_meta = lambda p, a='property': soup.find('meta', {a: p}).get('content') if soup.find('meta', {a: p}) else ""

        # Lấy ngày đăng và lọc lại lần cuối cho chắc chắn
        pub_date = get_meta('article:published_time')
        year = datetime.strptime(pub_date[:10], "%Y-%m-%d").year if pub_date else 0
        
        if not (START_YEAR <= year <= END_YEAR): 
            return None

        # Trích xuất nội dung văn bản (bỏ qua thẻ HTML)
        content_div = soup.find('div', {'data-role': 'content'}) or soup.find('div', class_='detail-content')
        content_text = content_div.get_text(separator='\n', strip=True) if content_div else ""

        # Trả về Dictionary đúng chuẩn yêu cầu
        return {
            "category": category,
            "sub_topic": sub_topic,
            "title": get_meta('og:title'),
            "description": get_meta('description', 'name') or get_meta('og:description'),
            "content": content_text,
            "public_date": pub_date,
            "author": get_meta('article:author', 'name'),
            "source": "Báo Thanh Niên",
            "url": url,
            "tag": get_meta('keywords', 'name')
        }
    except: 
        return None

def main():
    import re # Đừng quên import re ở đầu file
    total_articles = 0 

    # Đảm bảo thư mục đích luôn tồn tại (phòng hờ chạy ngoài Kaggle)
    os.makedirs('/kaggle/working/', exist_ok=True)

    for cat, subs in CATEGORIES.items():
        for sub, url in subs.items():
            sub_data = [] 
            
            # Xử lý chuỗi tạo tên file hợp lệ (thêm replace '/')
            safe_cat = cat.replace(' ', '_').replace('-', '_').replace('/', '_')
            safe_sub = sub.replace(' ', '_').replace('-', '_').replace('/', '_')
            
            # Gộp các dấu gạch dưới liên tiếp cho tên file đẹp hơn
            safe_cat = re.sub(r'_+', '_', safe_cat)
            safe_sub = re.sub(r'_+', '_', safe_sub)
            
            output_path = f'/kaggle/working/{safe_cat}_{safe_sub}.csv'

            print(f"\n>>> BẮT ĐẦU CHUYÊN MỤC: {cat.upper()} -> {sub.upper()}")
            links = get_all_links(url)
            print(f"  [+] Gom được {len(links)} link tiềm năng. Đang bóc tách nội dung chi tiết...")

            for i, link in enumerate(links):
                item = fetch_detail(link, cat, sub)
                if item:
                    sub_data.append(item)
                    total_articles += 1
                
                if (i+1) % 10 == 0: 
                    print(f"    - Đã xử lý {i+1}/{len(links)} link...")

            if sub_data:
                pd.DataFrame(sub_data).to_csv(output_path, index=False, encoding='utf-8-sig')
            
            print(f"  ✅ Đã lưu xong chuyên mục {cat.upper()} - {sub.upper()} vào file: {output_path}")

    print(f"\n🎉 HOÀN TẤT TẤT CẢ! Tổng cộng cào được {total_articles} bài báo hợp lệ.")

if __name__ == "__main__":
    main()


>>> BẮT ĐẦU CHUYÊN MỤC: DU LICH -> TIN TUC SU KIEN
  [*] Đang quét link tại: https://thanhnien.vn/du-lich/tin-tuc-su-kien.htm
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 29
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 62
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 102
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 142
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 182
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 221
    [Check] Năm cũ nhất đáy trang: 2026 | Tổng link đã gom: 261
    [Check] Năm cũ nhất đáy trang: 2025 | Tổng link đã gom: 301
    [Check] Năm cũ nhất đáy trang: 2025 | Tổng link đã gom: 341
    [Check] Năm cũ nhất đáy trang: 2025 | Tổng link đã gom: 381
    [Check] Năm cũ nhất đáy trang: 2025 | Tổng link đã gom: 421
    [Check] Năm cũ nhất đáy trang: 2025 | Tổng link đã gom: 461
    [Check] Năm cũ nhất đáy trang: 2025 | Tổng link đã gom: 501
    [Check] Năm cũ nhất đáy trang: 2025 | T